# compute_q80_deficits notebook

This notebook embeds the full code from `compute_q80_deficits.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

"""
Default simulated Q is **future** discharge
(Results/discharge_components_future/Q_total_all_basins.csv). Q80 remains the baseline
climatology from compute_seasonal_thresholds.py (historical model Q), so deficits compare
future flows to that reference.

Outputs (hydrological year starts 1 October by default):
  1) Optional daily long table (--save_daily).
  2) Per basin, per hydro-year: total sum(D), deficit-days, mean intensity sum(D)/deficit_days.
  3) Per basin, per meteorological season × season-year: same metrics.
     Season-year for DJF: December belongs to the winter named by the following January's year
     (e.g. Dec-2044 + Jan-2045 + Feb-2045 → DJF 2045).

CSV is the primary artifact. Use --plot for one PNG per basin (hydro-year + DJF/MAM/JJA/SON ΣD;
basin titles match other hydrographs). Use --plot_aggregate for an optional overview (mean across basins).
"""

from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from calibration_io import basin_hydrograph_title

SEASON_ORDER = ("DJF", "MAM", "JJA", "SON")
SEASON_COLORS = {"DJF": "#1f538d", "MAM": "#2d8659", "JJA": "#b35900", "SON": "#6c3483"}


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawTextHelpFormatter)
    p.add_argument("--root", type=Path, default=Path("."), help="Project root.")
    p.add_argument(
        "--q_csv",
        type=Path,
        default=None,
        help=(
            "Wide simulated Q CSV, columns date + basins (default: "
            "<root>/Results/discharge_components_future/Q_total_all_basins.csv). "
            "Use .../discharge_components/Q_total_all_basins.csv for historical model Q."
        ),
    )
    p.add_argument(
        "--q80_csv",
        type=Path,
        default=None,
        help="Q80 climatology wide table (ddmm rows; default: Results/thresholds/...).",
    )
    p.add_argument("--threshold_years", type=int, default=30, help="Suffix for default Q80 filename.")
    p.add_argument(
        "--basins",
        default="",
        help="Comma-separated basin IDs (default: all columns except date in Q CSV).",
    )
    p.add_argument(
        "--hydro_start_month",
        type=int,
        default=10,
        help="First month of hydrological year (default 10 = 1 Oct).",
    )
    p.add_argument(
        "--out_dir",
        type=Path,
        default=None,
        help="Output directory (default: <root>/Results/deficit_q80).",
    )
    p.add_argument(
        "--save_daily",
        action="store_true",
        help="Write daily long CSV (date, basin_id, q_sim, q80, D_mm, deficit_day); can be large.",
    )
    p.add_argument(
        "--plot",
        action="store_true",
        help="Save one PNG per basin under out_dir/plots/ (hydro-year + all seasons; titles like other hydrographs).",
    )
    p.add_argument(
        "--plot_aggregate",
        action="store_true",
        help="Also save one overview PNG: mean across basins (hydro-year + all seasons).",
    )
    return p.parse_known_args()[0]


def load_threshold_series(dates: pd.DatetimeIndex, wide: pd.DataFrame, basin_id: str) -> np.ndarray:
    if basin_id not in wide.columns:
        raise ValueError(f"Basin {basin_id!r} not in Q80 table columns: {list(wide.columns)[:10]}...")
    lut = wide.set_index("ddmm")[basin_id]
    ddmm = pd.Series(dates.strftime("%d-%m"), dtype=str)
    leap = (dates.month == 2) & (dates.day == 29)
    ddmm = ddmm.mask(leap, "28-02")
    s = ddmm.map(lut)
    s = pd.to_numeric(s, errors="coerce")
    return s.ffill().bfill().to_numpy(dtype=float)


def hydrological_year(dates: pd.Series | pd.DatetimeIndex, start_month: int) -> np.ndarray:
    """
    Integer label for hydrological year ending in September when start_month=10:
    Oct 2024–Sep 2025 → 2025 (year of the September closing the water year).
    """
    dt = pd.to_datetime(dates)
    y = dt.dt.year.to_numpy()
    m = dt.dt.month.to_numpy()
    # Year assigned = calendar year of Sep at end of Oct(start_month)–Sep block
    hy = np.where(m >= start_month, y + 1, y)
    return hy.astype(int)


def meteorological_season(month: int) -> str:
    if month in (12, 1, 2):
        return "DJF"
    if month in (3, 4, 5):
        return "MAM"
    if month in (6, 7, 8):
        return "JJA"
    return "SON"


def season_year(dates: pd.Series | pd.DatetimeIndex) -> np.ndarray:
    """DJF 2045 = Dec-2044, Jan-2045, Feb-2045 (label = year of Jan/Feb)."""
    dt = pd.to_datetime(dates)
    y = dt.dt.year.to_numpy()
    m = dt.dt.month.to_numpy()
    sy = np.where(m == 12, y + 1, y)
    return sy.astype(int)


def deficit_block(
    dates: pd.DatetimeIndex,
    q_sim: np.ndarray,
    q80: np.ndarray,
    basin_id: str,
) -> pd.DataFrame:
    q_sim = np.asarray(q_sim, dtype=float)
    q80 = np.asarray(q80, dtype=float)
    valid = np.isfinite(q_sim) & np.isfinite(q80)
    d = np.maximum(0.0, q80 - q_sim)
    d = np.where(valid, d, np.nan)
    below = (q_sim < q80) & valid
    return pd.DataFrame(
        {
            "date": dates,
            "basin_id": basin_id,
            "q_sim": q_sim,
            "q80": q80,
            "D_mm": d,
            "deficit_day": below.astype(np.int8),
        }
    )


def aggregate_hydro(daily: pd.DataFrame, start_month: int) -> pd.DataFrame:
    x = daily.copy()
    x["hydro_year"] = hydrological_year(x["date"], start_month)
    g = x.groupby(["basin_id", "hydro_year"], sort=True)
    sum_d = g["D_mm"].sum(min_count=1)
    days = g["deficit_day"].sum()
    n_days = x.groupby(["basin_id", "hydro_year"])["D_mm"].apply(lambda s: int(s.notna().sum()))
    out = pd.DataFrame({"sum_D_mm": sum_d, "deficit_days": days, "n_days": n_days})
    out = out.reset_index()
    out["mean_deficit_intensity_mm_d"] = np.where(
        out["deficit_days"] > 0,
        out["sum_D_mm"] / out["deficit_days"],
        np.nan,
    )
    return out


def aggregate_season_year(daily: pd.DataFrame) -> pd.DataFrame:
    x = daily.copy()
    x["month"] = x["date"].dt.month
    x["season"] = x["month"].map(meteorological_season)
    x["season_year"] = season_year(x["date"])
    g = x.groupby(["basin_id", "season", "season_year"], sort=True)
    sum_d = g["D_mm"].sum(min_count=1)
    days = g["deficit_day"].sum()
    n_days = x.groupby(["basin_id", "season", "season_year"])["D_mm"].apply(lambda s: int(s.notna().sum()))
    out = pd.DataFrame({"sum_D_mm": sum_d, "deficit_days": days, "n_days": n_days})
    out = out.reset_index()
    out["mean_deficit_intensity_mm_d"] = np.where(
        out["deficit_days"] > 0,
        out["sum_D_mm"] / out["deficit_days"],
        np.nan,
    )
    return out


def _plot_year_series(
    ax,
    idx: np.ndarray,
    vals: np.ndarray,
    color: str,
    title: str,
    ylabel: str,
    *,
    show_xlabel: bool = True,
) -> None:
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    if len(idx) > 45:
        ax.plot(idx, vals, color=color, linewidth=1.2)
        if show_xlabel:
            ax.set_xlabel("Year")
    else:
        ax.bar(idx.astype(str), vals, color=color, width=0.8)
        ax.tick_params(axis="x", rotation=45)
        if show_xlabel:
            ax.set_xlabel("Year")
    if not show_xlabel:
        ax.tick_params(axis="x", labelbottom=False)


def save_basin_deficit_plot(hydro_b: pd.DataFrame, season_b: pd.DataFrame, basin_id: str, out_png: Path) -> None:
    """Hydro-year ΣD plus meteorological seasons DJF / MAM / JJA / SON for one basin."""
    out_png.parent.mkdir(parents=True, exist_ok=True)
    hy = hydro_b.sort_values("hydro_year")

    fig, axes = plt.subplots(5, 1, figsize=(10, 14), sharex=False)
    fig.suptitle(
        basin_hydrograph_title(basin_id)
        + "\nΣD = Σ max(0, Q80 − Q_future); Q80 = baseline seasonal climatology",
        fontsize=10,
        y=0.995,
    )

    _plot_year_series(
        axes[0],
        hy["hydro_year"].to_numpy(),
        hy["sum_D_mm"].to_numpy(),
        "#2c5282",
        "Hydrological year — total ΣD (mm)",
        "ΣD (mm)",
        show_xlabel=False,
    )

    for i, sea in enumerate(SEASON_ORDER, start=1):
        sub = season_b.loc[season_b["season"] == sea].sort_values("season_year")
        _plot_year_series(
            axes[i],
            sub["season_year"].to_numpy(),
            sub["sum_D_mm"].to_numpy(),
            SEASON_COLORS[sea],
            f"{sea} — seasonal ΣD (mm)",
            "ΣD (mm)",
            show_xlabel=(i == 4),
        )

    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(out_png, dpi=150)
    plt.close(fig)


def save_aggregate_plot(hydro: pd.DataFrame, season: pd.DataFrame, out_png: Path) -> None:
    """Mean across basins: hydro-year and each meteorological season."""
    out_png.parent.mkdir(parents=True, exist_ok=True)
    hy_mean = hydro.groupby("hydro_year", sort=True)["sum_D_mm"].mean()

    fig, axes = plt.subplots(5, 1, figsize=(10, 14), sharex=False)
    fig.suptitle(
        "Mean across basins — ΣD = Σ max(0, Q80 − Q_future)\n"
        "Q80 = baseline seasonal climatology; Q_future = scenario discharge",
        fontsize=10,
        y=0.995,
    )

    _plot_year_series(
        axes[0],
        hy_mean.index.to_numpy(),
        hy_mean.values,
        "#2c5282",
        "Hydrological year — mean ΣD (mm)",
        "Mean ΣD (mm)",
        show_xlabel=False,
    )

    for i, sea in enumerate(SEASON_ORDER, start=1):
        sub = season.loc[season["season"] == sea].copy()
        sea_mean = sub.groupby("season_year", sort=True)["sum_D_mm"].mean()
        _plot_year_series(
            axes[i],
            sea_mean.index.to_numpy(),
            sea_mean.values,
            SEASON_COLORS[sea],
            f"{sea} — mean seasonal ΣD (mm)",
            "Mean ΣD (mm)",
            show_xlabel=(i == 4),
        )

    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(out_png, dpi=150)
    plt.close(fig)


def main() -> None:
    args = parse_args()
    root = args.root.resolve()
    q_csv = args.q_csv or (root / "Results" / "discharge_components_future" / "Q_total_all_basins.csv")
    q80_csv = args.q80_csv or (
        root / "Results" / "thresholds" / f"Q80_daily_thresholds_last{args.threshold_years}y.csv"
    )
    out_dir = args.out_dir or (root / "Results" / "deficit_q80")
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not q_csv.exists():
        raise FileNotFoundError(f"Q CSV not found: {q_csv}")
    if not q80_csv.exists():
        raise FileNotFoundError(f"Q80 CSV not found: {q80_csv}")

    print(f"Simulated Q (deficit vs Q80): {q_csv.resolve()}")
    print(f"Q80 climatology: {q80_csv.resolve()}")

    q_df = pd.read_csv(q_csv)
    if "date" not in q_df.columns:
        raise ValueError("Q CSV must have a 'date' column.")
    q_df["date"] = pd.to_datetime(q_df["date"], errors="coerce").dt.normalize()
    q_df = q_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

    all_basins = [c for c in q_df.columns if c != "date"]
    if args.basins.strip():
        basin_ids = [b.strip() for b in args.basins.split(",") if b.strip()]
        for b in basin_ids:
            if b not in q_df.columns:
                raise ValueError(f"Basin {b!r} not in Q CSV columns.")
    else:
        basin_ids = all_basins

    q80_wide = pd.read_csv(q80_csv)
    if "ddmm" not in q80_wide.columns:
        raise ValueError("Q80 wide table must have 'ddmm' column.")

    dates = pd.DatetimeIndex(q_df["date"].values)
    daily_parts: list[pd.DataFrame] = []
    for basin_id in basin_ids:
        q80_arr = load_threshold_series(dates, q80_wide, basin_id)
        q_sim = pd.to_numeric(q_df[basin_id], errors="coerce").to_numpy(dtype=float)
        daily_parts.append(deficit_block(dates, q_sim, q80_arr, basin_id))

    daily_long = pd.concat(daily_parts, ignore_index=True)

    hydro = aggregate_hydro(daily_long, int(args.hydro_start_month))
    season_tbl = aggregate_season_year(daily_long)

    hydro_path = out_dir / "deficits_hydro_year_by_basin.csv"
    season_path = out_dir / "deficits_season_year_by_basin.csv"
    hydro.to_csv(hydro_path, index=False)
    season_tbl.to_csv(season_path, index=False)

    if args.save_daily:
        daily_path = out_dir / "deficits_daily_long.csv"
        daily_long.to_csv(daily_path, index=False)
        print(f"Saved: {daily_path}")

    print(f"Hydro year starts month {args.hydro_start_month} (1=Jan).")
    print(f"Saved: {hydro_path}")
    print(f"Saved: {season_path}")

    if args.plot:
        plot_dir = out_dir / "plots"
        plot_dir.mkdir(parents=True, exist_ok=True)
        for basin_id in basin_ids:
            hb = hydro.loc[hydro["basin_id"] == basin_id]
            sb = season_tbl.loc[season_tbl["basin_id"] == basin_id]
            out_png = plot_dir / f"deficit_future_{basin_id}.png"
            save_basin_deficit_plot(hb, sb, basin_id, out_png)
        print(f"Saved per-basin figures under: {plot_dir}")

    if args.plot_aggregate:
        agg_path = out_dir / "deficit_future_mean_across_basins.png"
        save_aggregate_plot(hydro, season_tbl, agg_path)
        print(f"Saved: {agg_path}")


# Execute the script entry point
main()
